In [6]:
import numpy as np
import matplotlib.pyplot as plt
import random
import copy
import hydra
from hydra import initialize, compose
from omegaconf import DictConfig

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict

from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

In [7]:
from os.path import join, dirname
import os
import sys
import time

import torch
from torch import nn, einsum
import torch.nn.functional as F
from torch.distributions import Normal

from einops import rearrange, repeat

sys.path.append(join(dirname(dirname(os.getcwd()))))
sys.path.append(os.getcwd())

from memoryPerceiver.memory_perceiver.main.configuration import cf

from memoryPerceiver.memory_perceiver.core.model_net import PerceiverIO
from memoryPerceiver.memory_perceiver.processing.vectorize import input_to_tensor, fourier_encode

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: ISAAC_JUPYTER_KERNEL=1


In [8]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device_cpu = torch.device("cpu")
random.seed(0)
np.random.seed(0)
torch.use_deterministic_algorithms(False)
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

env: CUBLAS_WORKSPACE_CONFIG=:4096:8


In [9]:
nb_frames = 32
frames_future = 8
len_video_frame = 1824  # 456*256/(8*8)
len_audio_frame = 120   # 1920/16
query_dim = 832
video_latent_dim = 256
audio_latent_dim = 560
batch_size = 1

len_vid = nb_frames*len_video_frame
len_audio = nb_frames*len_audio_frame
len_audio_raw = len_audio*cf.sample_per_frame
video_future = frames_future*len_video_frame
audio_future = frames_future*len_audio_frame

special_compression = [i for i in range(video_future)] + [i for i in range(len_vid, len_vid+audio_future)]
for k in range(len(cf.body_parts)):
    special_compression += [ len_vid + len_audio + k*nb_frames + i for i in range(frames_future) ]

In [10]:
def create_model( device=device):

    # Projection to return modalities, enhancement

    modality_parameters = {}
    modality_parameters['audio'] = audio_latent_dim
    modality_parameters['video'] = video_latent_dim
    for part, params in cf.body_parts.items():
        modality_parameters[part] = query_dim - 2*128 - params
    for part, params in cf.body_outputs.items():
        modality_parameters[part] = query_dim - 2*128 - params

    projection_parameters = dict(cf.body_outputs)
    projection_parameters.update({'audio': 16})

    # Instantiate model
        
    model = PerceiverIO(
            2,
            query_dim,
            special_compression = special_compression,
            device=device,
            projection_parameters = projection_parameters,
            modality_parameters = modality_parameters,
            max_batch_size = batch_size
        )
    model = model.to(device)

    # Queries

    queries = []
    audio_query = repeat( torch.cat([model.modality['audio'], torch.zeros(16, device=device)]), '... -> n ...', n=audio_future)
    audio_pos = fourier_encode(torch.linspace(-1, 1, audio_future), audio_future/2, num_bands=128).to(device)
    queries.append(torch.cat([audio_query, audio_pos], dim=-1).to(device))
    for part, params in cf.body_outputs.items():
        part_query = repeat( torch.cat([model.modality[part], torch.zeros(params, device=device)]), '... -> n ...', n=frames_future)
        part_pos = fourier_encode(torch.linspace(-1, 1, frames_future), frames_future/2, num_bands=128).to(device)
        queries.append(torch.cat([part_query, part_pos], dim=-1).to(device))
    model.queries=torch.cat(queries)


    return model

In [11]:
model = create_model()
model_student = create_model()
model_critic = nn.Sequential(
                        nn.Linear(4, 64),
                        nn.ReLU(),
                        nn.Linear(64, 64),
                        nn.ReLU(),
                        nn.Linear(64, 64),
                        nn.ReLU(),
                        nn.Linear(64, 1)
                    ).to(device)

# inputs={}
# inputs['video']=torch.randn(batch_size, 3, nb_frames, 256, 456)
# inputs['audio']=torch.randn(batch_size, nb_frames*1920)
# inputs['observations']=torch.randn(batch_size, nb_frames, 4)
# inputs['expectation']=torch.randn(batch_size, nb_frames, 1)
# batch = input_to_tensor(inputs, model.modality, cf.body_parts).to(device)
# model(batch, queries = model.queries, update_mem=False)

In [7]:
#@hydra.main(config_name="config", config_path="../cfg")
#@hydra.main(config_name="config", config_path="../../OmniIsaacGymEnvs/omniisaacgymenvs/cfg")
def parse_hydra_configs(cfg: DictConfig):

    def buffer_padding(buffer, dim):
        # zero padding then transform: t b (d) -> b t (d)
        pad0 = np.max([0, nb_frames-len(buffer)])
        res = torch.swapaxes( 
                torch.cat([ torch.zeros([pad0, numEnvs]+dim, device=device), torch.stack(buffer) ])
                , 0, 1)
        return res

    def update_buffer(buff, key, element):
        buff[key].append(element)
        if(len(buff[key])>nb_frames):
            buff[key].pop(0)

    cfg_dict = omegaconf_to_dict(cfg)

    headless = cfg.headless
    render = not headless
    render = True

    env = VecEnvRLGames(headless=headless, sim_device=cfg.device_id)
    # sets seed. if seed is -1 will pick a random one
    from omni.isaac.core.utils.torch.maths import set_seed
    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed
    task = initialize_task(cfg_dict, env)

    history = {'video':[], 'audio':[], 'obs':[], 'reward':[], 'action':[] ,'returns':[], 'values': []}
    buffer = {'video':[], 'audio':[], 'obs':[], 'reward':[], 'action':[], 'returns':[], 'expectation_goal':[]}
    replay_training=True  # Will switch to false immediately
    memories = {i:None for i in range(numEnvs)}
    rewards_mem = torch.tensor([], device=device)
    death_index = {i:[] for i in range(numEnvs)}
    cumulative_reward = torch.zeros(numEnvs, 1, device=device)
    last_batch2=None
    first_run=True
    all_losses=[]
    crit_losses=[]
    c_gammas = rearrange(c_gamma ** torch.arange(0, sim_duration, device=device), 't -> t 1')
    c_lambdas_gammas = rearrange((c_lambda*c_gamma) ** torch.arange(0, sim_duration, device=device), 't -> t 1')
    sim_count = 0
    returns_mean, returns_m2, returns_std = 5.5, 110, 11
    while env._simulation_app.is_running():
        if env._world.is_playing():
            # if env._world.current_time_step_index == 0 or \
            # env._world.current_time_step_index % sim_duration == 0:
            if env.sim_frame_count % sim_duration == 0:

                ### Initialization

                if sim_count % 4 == 1:
                    plt.plot(all_losses, label='mod')
                    plt.plot(crit_losses, label='crit')
                    plt.savefig('/isaac-sim/memoryPerceiver/memory_perceiver/main/loss.png')
                env._world.reset(soft=False)
                replay_training = not replay_training
                # print('$$$$$$$$$$$ CUMUL REWARD $$$$$$$$$$$')
                # print(cumulative_reward)
                if replay_training:
                    print('REPLAY')
                    model.load_state_dict(torch.load(model_path))
                    model.reset_memories()
                    model_student.reset_memories()
                    model.eval()
                    model_student.train() 
                    optimizer = torch.optim.Adam(model_student.parameters(), lr=learning_rate)
                    optimizer_critic = torch.optim.Adam(model_critic.parameters(), lr=learning_rate_critic)
                    # Running returns metrics
                    if history['returns'] != [] :
                        returns_hist = torch.stack(history['returns'])
                        returns_mean = returns_mean*c_gamma_returns + returns_hist.mean()*(1-c_gamma_returns)
                        returns_m2 = returns_m2*c_gamma_returns + ((returns_hist-returns_mean) ** 2).mean()*(1-c_gamma_returns)
                        returns_std = torch.sqrt(torch.abs(returns_m2))
                    history['values'].append(torch.zeros(numEnvs, 1 ,device=device))
                    values_mem = torch.stack(history['values'])
                    rewards_mem = torch.stack(history['reward'])
                    first_run=False
                else:
                    print('PLAY')
                    if first_run and os.path.exists(model_path):
                        model.load_state_dict(torch.load(model_path))
                    else:
                        if sim_count % student_update_step*2 == 0:
                            model.load_state_dict(model_student.state_dict())
                    if first_run and os.path.exists(critic_path):
                        model_critic.load_state_dict(torch.load(critic_path))
                    model.reset_memories()
                    torch.save(model.state_dict(), model_path)
                    torch.save(model_critic.state_dict(), critic_path)
                    model.eval()
                    death_index = {i:[] for i in range(numEnvs)}
                    
                torch.manual_seed(sim_count//2)
                memories = {i:None for i in range(numEnvs)}
                cumulative_reward = torch.zeros(numEnvs, 1, device=device)
                expectation_goal = initial_expectation*torch.ones(numEnvs, device=device)
                for key in buffer.keys():
                    buffer[key]=[]
                for key in history.keys():
                    history[key]=history[key][-sim_duration:]
                backward_count=0
                sim_count+=1
                sim_step = 0
            sim_step+=1

            ### Observations: 'cartpole_view' 'obs_buf' (numEnvs, 4)

            all_obs = env._task.get_observations()['cartpole_view']
            obs = all_obs['obs_buf'].clone()
            update_buffer(buffer, 'obs', obs)
            history['obs'].append(obs)

            ### Video: 'cartpole_view' 'obs_buf' (numEnvs, 3, 256, 456)

            vid = all_obs['video_buf'].clone()
            update_buffer(buffer, 'video', vid)

            ### Reward (numEnvs, 1)

            env._task.calculate_metrics()
            reward=env._task.rew_buf.clone()
            update_buffer(buffer, 'reward', reward)
            history['reward'].append(reward)

            ### Expectation/ Return

            cumulative_reward += torch.sum(reward, dim=-1)
            returns_tmp=[]
            advantages_tmp=[]
            expectation_goal -= reward

            for agent_i in range(numEnvs):

                # Reset goal and memory when dying

                if env._task.reset_buf[agent_i].item() == 1:
                    expectation_goal[agent_i] = initial_expectation
                    memories[agent_i] = None

                    # Update death dictionnary during inference

                    if not replay_training:
                        death_index[agent_i].append(sim_step)

                # Compute expectation and advantages with death dictionnary during training

                if replay_training: 
                    if len(death_index[agent_i])>0 :
                        if sim_step > death_index[agent_i][0] :
                            death_index[agent_i].pop(0)
                    death_i = death_index[agent_i][0] if len(death_index[agent_i]) > 0 else sim_duration
                    returns_tmp.append([torch.sum(rewards_mem[sim_step:death_i, agent_i]*c_gammas[:death_i-sim_step]).item()])
                    TDs = rewards_mem + c_gamma*values_mem[1:] - values_mem[:-1]
                    advantages_tmp.append([torch.sum(TDs[sim_step:death_i, agent_i]*c_lambdas_gammas[:death_i-sim_step]).item()])

            update_buffer(buffer, 'expectation_goal', rearrange(expectation_goal, ' d -> d 1'))
            if replay_training: 
                returns_tmp=torch.tensor(returns_tmp, device=device)
                history['returns'].append(returns_tmp)
                update_buffer(buffer, 'returns', returns_tmp)
                update_buffer(buffer, 'advantages', torch.tensor(advantages_tmp, device=device))

            ### Process input

            inputs={}
            inputs['video'] = torch.zeros(numEnvs, 3, nb_frames, 256, 456, device=device)
            # inputs['video'] = rearrange(buffer_padding(buffer['video'], [3, 256, 456]), 'b t c h w -> b c t h w')
            inputs['audio'] = torch.zeros(numEnvs, nb_frames*1920, device=device)
            if len(history['action']) == 0:
                history['action'].append(torch.zeros(numEnvs, cf.body_outputs['actions'], device=device))
            inputs['past_actions'] = buffer_padding(history['action'][-nb_frames:], [cf.body_outputs['actions']])
            inputs['observations'] = buffer_padding(buffer['obs'], [cf.body_parts['observations']])


            ### Agent
            
            new_actions = []
            returns, values = None, None
            if len(buffer['action'])==0 :
                # returns = buffer_padding(buffer['expectation_goal'], [1])
                batch = input_to_tensor(inputs, model.modality, cf.body_parts)

                values = model_critic(inputs['observations'][:,-8:])

                if replay_training:
                    # returns = (returns - returns_mean) / (returns_std + eps)
                    batch_training = input_to_tensor(inputs, model.modality, cf.body_parts)

                    ## Critic

                    returns = buffer_padding(buffer['returns'], [1])[:,-8:]
                    advantages = buffer_padding(buffer['advantages'], [1])[:,-8:]
                    # advantages = (returns - values).detach()
                    # advantages = (rewards + c_gamma*values1 - values).detach()
                    critic_loss = F.l1_loss(values, returns)
                    critic_loss.backward()
                    optimizer_critic.step()
                    optimizer_critic.zero_grad()
                    crit_losses.append(critic_loss.item())


                for agent_i in range(numEnvs):

                    ## Training model inference before memory update

                    if replay_training:
                        if(memories[agent_i] is not None):
                            model_student.load_memories(memories[agent_i])
                        else:
                            model_student.reset_memories()
                        output_training = {}
                        x_training, aux_loss = model_student(batch_training[agent_i:agent_i+1], 
                                                            queries = model_student.queries, update_mem=False)
                        output_training['audio'] = rearrange(model_student.projections['audio'](
                            x_training[:, :audio_future, :]), "b t t1 -> b (t t1)" )
                        start_modality = audio_future
                        for part, params in cf.body_outputs.items():
                            output_training[part] = model_student.projections[part](
                                x_training[:, start_modality:start_modality+frames_future, :] )
                            start_modality += frames_future

                    ## Agent inference and memory update
                               
                    output = {}
                    if(memories[agent_i] is not None):
                        model.load_memories(memories[agent_i])
                    else:
                        model.reset_memories()

                    with torch.no_grad():
                        x, _ = model(batch[agent_i:agent_i+1], queries = model.queries)
                        output['audio'] = rearrange(model.projections['audio'](
                            x[:, :audio_future, :]), "b t t1 -> b (t t1)" )
                    start_modality = audio_future
                    for part, params in cf.body_outputs.items():
                        output[part] = model.projections[part](
                            x[:, start_modality:start_modality+frames_future, :] )
                        start_modality += frames_future
                    actions = torch.tanh(output['actions'])[0]
                    past_actions=inputs['past_actions'][agent_i, -8:].detach()
                    mean, stddev = actions[0], actions[1:].std(dim=0)
                    shift=(actions[1:]-mean).mean(dim=0)
                    dist = Normal(mean, stddev)
                    output['logprobs'] = dist.log_prob(past_actions)
                    sample, interpol= dist.sample().detach(), torch.linspace(0, 1, frames_future, device=device)
                    output['actions'] = einsum('i, j -> ij', interpol, sample) + \
                                        einsum('i, j -> ij', 1-interpol, sample+shift)
                    new_actions.append(output['actions'])
                    memories[agent_i] = model.return_memories(device=device)

                    ## Compute loss

                    if replay_training:

                        ## Student

                        # !!!!!!!! Warning, delete 0 later !!!!!!
                        loss = aux_loss * 0
                        start_modality = audio_future
                        #for part, params in cf.body_outputs.items():
                        actions = torch.tanh(output_training['actions'])[0]
                        mean, stddev = actions[0], actions[1:].std(dim=0)
                        dist = Normal(mean, stddev)

                        # Statistics
                        logprobs = dist.log_prob(past_actions)
                        entropy = dist.entropy()
                        advantage = advantages[agent_i]

                        # Loss
                        ratios = torch.exp(logprobs - output['logprobs'].detach())
                        surr1 = ratios * advantage
                        surr2 = torch.clamp(ratios, 1-eps_clip, 1+eps_clip) * advantage
                        loss = -torch.min(surr1, surr2) - c_entropy * entropy 
                        loss = loss.mean()/(numEnvs*accumulation_steps)
                        loss.backward(retain_graph=True)
                        backward_count += 1


                # Step optimizer

                if replay_training:
                    if backward_count % (numEnvs*accumulation_steps) == 0 and backward_count != 0:
                        optimizer.step()
                        optimizer.zero_grad()
                        backward_count = 0
                    all_losses.append(loss.item())

                # Update buffers

                new_actions = list(torch.unbind( rearrange(new_actions, 'b t d -> t b d') ))
                buffer['action'] += new_actions
                history['action'] += new_actions
                history['values'] += list(torch.unbind( rearrange(values, 'b t () -> t b ()') ))


            ## Print

            # Current action (numEnvs, 1)

            if(len(buffer['action'])>0):
                action = buffer['action'].pop(0)
                action = torch.clamp(action, min=-1, max=1)
            else:
                action = torch.dzeros(numEnvs, 1)

            if( returns is not None):
                print('$$$$$$$$$$$')
                print('mean:', returns_mean)
                print('std:', returns_std)
                print('actions: ', past_actions)
                print('returns: ', returns)
                print('values: ', values)

            # if env.sim_frame_count % sim_duration == 1:
            #     print("$$$$$$$$$$$$ 00000 $$$$$$$$$$$")
            #     if replay_training:
            #         print('returns: ', returns)
            #         print('action: ', action)
            #         print('observations',inputs['observations'][-1][-1])
            #         print('loss', np.mean(all_losses[-100:]))
            
            # if env.sim_frame_count % sim_duration == 50:
            #     print("$$$$$$$$$$$$ 11111 $$$$$$$$$$$")
            #     if replay_training:
            #         print('returns: ', returns)
            #         print('action: ', action)
            #         print('observations',inputs['observations'][-1][-1])
            #         print('loss', np.mean(all_losses[-100:]))

            # if env.sim_frame_count % sim_duration == 99:
            #     print("$$$$$$$$$$$$ 22222 $$$$$$$$$$$")
            #     if replay_training:
            #         print('returns: ', returns)
            #         print('action: ', action)
            #         print('observations',inputs['observations'][-1][-1])
            #         print('loss', np.mean(all_losses[-100:]))
            #     if last_batch2 is not None :
            #         print(torch.sum(last_batch2 - model.queries))
            #     last_batch2=model.queries

            # action = 0.1*torch.ones(numEnvs, 1)

            # Step environment

            env._task.pre_physics_step(action.to(task.rl_device))
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()

        else:
            env._world.step(render=render)

    env._simulation_app.close()


if __name__ == '__main__':
    sim_duration = 104
    numEnvs = 2
    learning_rate = 0.0001
    learning_rate_critic = 0.0003
    initial_expectation = 200
    accumulation_steps = 5
    c_entropy = 0.01
    student_update_step = 4
    eps=1e-10
    c_gamma=0.99
    c_lambda=0.9
    c_gamma_returns=0.9
    eps_clip=0.1
    model_path = 'weights/cartpole.pt'
    critic_path = 'weights/critic.pt'
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Cartpole", "headless=True","+numEnvs="+str(numEnvs)])
    parse_hydra_configs(cfg)

/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:412: UserWarning: In config: Invalid overriding of hydra/job_logging:
Default list overrides requires 'override' keyword.
See https://hydra.cc/docs/next/upgrades/1.0_to_1.1/defaults_list_override for more information.

  deprecation_warning(msg)


Starting kit application with the following args:  ['/isaac-sim/exts/omni.isaac.kit/omni/isaac/kit/simulation_app.py', '/isaac-sim/apps/omni.isaac.sim.python.gym.headless.render.kit', '--/app/tokens/exe-path=/isaac-sim/kit', '--/persistent/app/viewport/displayOptions=3094', '--/rtx/materialDb/syncLoads=True', '--/rtx/hydra/materialSyncLoads=True--/omni.kit.plugin/syncUsdLoads=True', '--/app/renderer/resolution/width=1280', '--/app/renderer/resolution/height=720', '--/app/window/width=1440', '--/app/window/height=900', '--/renderer/multiGpu/enabled=True', '--/app/fastShutdown=False', '--ext-folder', '/isaac-sim/exts', '--ext-folder', '/isaac-sim/apps', '--/physics/cudaDevice=0', '--portable', '--no-window', '--allow-root']
Passing the following args to the base kit application:  ['-f', '/root/.local/share/jupyter/runtime/kernel-8cd8e563-df13-48c2-b3a0-00e25d85f5ef.json']
[Warning] [omni.isaac.kit.simulation_app] Modules: ['omni.isaac.kit.app_framework'] were loaded before SimulationApp 

/isaac-sim/kit/kernel/py/omni/kit/app/_impl/__init__.py:104: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated since Python 3.3,and in 3.9 it will stop working
  if not isinstance(setting_py, str) and isinstance(setting_py, collections.Sequence):
/isaac-sim/kit/exts/omni.graph/omni/graph/core/_impl/autonode/type_definitions.py:12: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  Float = NewType("Float", np.float)


] [ext: omni.kit.window.filepicker-2.7.15] startup
[2.339s] [ext: omni.kit.widget.settings-1.0.1] startup
[2.348s] [ext: omni.kit.menu.utils-1.4.7] startup
[2.356s] [ext: omni.rtx.window.settings-0.6.2] startup
[2.363s] [ext: omni.hydra.pxr-1.1.2] startup
[2.372s] [ext: omni.hydra.pxr.settings-1.0.3] startup
[2.373s] [ext: omni.kit.viewport.pxr-104.0.1] startup
[2.373s] [ext: omni.convexdecomposition-104.2.4-5.1] startup
[2.376s] [ext: omni.iray.libs-0.0.0] startup
[2.385s] [ext: omni.volume-0.1.0] startup
[2.389s] [ext: omni.mdl.neuraylib-0.1.0] startup
[2.393s] [ext: omni.kvdb-0.0.0] startup
[2.395s] [ext: omni.hydra.rtx-0.1.0] startup
[2.407s] [ext: omni.localcache-0.0.0] startup
[2.410s] [ext: omni.kit.viewport.rtx-104.0.0] startup
[2.410s] [ext: omni.isaac.kit-1.4.1] startup
[2.411s] [ext: omni.hydra.engine.stats-1.0.0] startup
[2.416s] [ext: omni.kit.collaboration.channel_manager-1.0.9] startup
[2.419s] [ext: omni.kit.window.file_importer-1.0.10] startup
[2.421s] [ext: omni.kit.c

/isaac-sim/kit/python/lib/python3.7/site-packages/gym/spaces/box.py:84: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


ni.isaac.core-1.46.3] startup
[3.829s] [ext: omni.warp-0.6.3] startup
[4.182s] [ext: omni.kit.window.title-1.1.2] startup
2023-08-03 00:59:24 [4,172ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2023-08-03 00:59:24 [4,172ms] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
2023-08-03 00:59:24 [4,173ms] [Warning] [carb] [Plugin: libomni.structuredlog.plugin.so] Module /isaac-sim/kit/libomni.structuredlog.plugin.so remained loaded after unload request
2023-08-03 00:59:24 [4,175ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2023-08-03 00:59:24 [4,175ms] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
[4.189s] [ext: omni.kit.viewport.bundle-104.0.2] startup
[4.189s] [e